In [ ]:
# -*- coding: utf-8 -*-
from src.file_tools import saveFile
from src.model_tools import generate_response, get_model_and_tokenizer
from src.parsing_tools import extract_and_parse_json_array, format_topic_as_md, format_topic
from src.md_latex_tools import fix_latex
# from src.run_tools import run_with_retries
import datetime
import os

In [ ]:
folder_path = os.path.join(os.path.expanduser("~"), "LLM-Model")
config_dict = {
    "model_name": "Qwen/Qwen2.5-7B-Instruct",
    "model_dir": folder_path,
    "device_config": "auto",  # "auto", "cpu", or "mps"
    "torch_dtype": "auto",  # "auto" or specific type
}

# Initialize once and reuse
model, tokenizer = get_model_and_tokenizer(config_dict)

# Get the topics from the user
topic = "Logstash"
directory = topic+"_"+config_dict.get("model_name").split(
    "/")[1]+datetime.datetime.now().strftime("%Y-%m-%d")

In [3]:
main_prompt = f"""Answer the following request concisely without using filler words or sentences like 'sure ..', 'certainly ..', or 'of course'.
Generate a comprehensive list of key topics needed to learn: {topic}.
I only nedd the main topics and there subtopic.
format the response like this: 
1. topic 1 : add short description 
    - subtopic 1: short description 
    - subtopic 2: short description 
    ...
2. topic 2 : add short description 
    - subtopic 1: short description 
    - subtopic 2: short description 
    - subtopic 3: short description 
    ...

Dont forget to add the short description for each topic and subtopic.
"""

main_topics = generate_response(main_prompt, tokenizer, model)

: 

: 

In [ ]:
main_prompt = f"""
can you have a look at the my list of topics and subtopics and extend and apply improvement to it?
Keep the format of the list as it is.
here is the list to the topic {topic}:
{main_topics}
"""

main_topics_enhanced = generate_response(main_prompt, tokenizer, model)

print(main_topics_enhanced)

In [ ]:
main_prompt = f""" you will be given a text that contains a list of topics and theres subtopics.
you need to format the text in a json format.
each one of topic and subtopic are important and should be included in the json format. dont forget any key.
Present the list in json format like this example for a topic on Linear Algebra:
[
  {{
    "number": 1, # obligatory
    "topic": "Linear Equations", # obligatory
    "description": "Understanding how to solve systems of linear equations using methods such as substitution or elimination.", # obligatory
    "subtopics": ["Solving Linear Systems:description", "Using Matrices:description", "Inverse Matrices:description"] # obligatory
  }},
  {{
    "number": 2,
    "topic": "...",
    "description": "....",
    "subtopics": ["..", "..", ".."]
  }},
  ...
]
here is the text:
{main_topics_enhanced}
"""


def run_with_retries(max_retries=3):
    attempts = 0
    while attempts < max_retries:
        try:
            response = generate_response(main_prompt, tokenizer, model)
            json = extract_and_parse_json_array(response)
            saveFile(directory=directory, fileName="Topics",
                     content=format_topic_as_md(json, topic))
            print("Function executed successfully!")
            return json  # If successful, return the result
        except Exception as e:
            # Handle the exception (print or log it) and retry
            attempts += 1
            print(f"Attempt {attempts} failed: {e}")
            if attempts == max_retries:
                print("Max retries reached. Giving up.")
                raise  # Re-raise the last exception after max retries


json = run_with_retries(max_retries=3)

In [ ]:
# Generate detailed explanations for each topic

for index, item in enumerate(json):
    explanation_prompt = f"""
    I am learing about "{topic}" and i have already list of key topic for it.
    I will provide you with my list, but i only want you tp provide a long detailed comprehensive explanation for one of them, which is '{format_topic(item)}' in relation to the topic: {topic}.
    The other topics i will handle separately in the futur.
    The topic and subtopics should numbered depending on the order in the list.
    here are main topics and subtopics:
    {json}
        """
    content = generate_response(
        explanation_prompt, tokenizer=tokenizer, model=model, max_new_tokens=4096)
    saveFile(directory, str(index+1)+"- "+item.get("topic", ""), content)

In [ ]:
# step 3: fix latex formatting
fix_latex(
    path="./"+topic,  # Replace with your folder path
    show_changes=True,  # Set to True to log changes
    overwrite_original_file=True,  # Set to True to overwrite original files
)

In [ ]:
# convert md to pdf
!find ./ -name "*.md" -exec pandoc {} \
--pdf-engine=/Library/TeX/texbin/pdflatex \
-V geometry:margin=0.5cm \
-V fontsize=12pt \
-V documentclass=article \
-o {}.pdf \;